<a href="https://colab.research.google.com/github/Ngwijabagabo/Groundwater_Storage_pred/blob/main/GWPZ_RF_MaxEnt_EasternRwanda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Groundwater potential, Eastern Province (Rwanda): Random Forest and MaxEnt, current conditions and CMIP6 rainfall scenarios
**Analysis code for:** Ngwijabagabo, H., Kabera, T., Nyandwi, E., Uwanyirigira, J. (2026). *Modeling groundwater potential under climate change scenarios in Eastern Rwanda using the MaxEnt and RF machine learning algorithms.* Physics and Chemistry of the Earth (JPCE-D-26-00483). Data and outputs: Zenodo, https://doi.org/10.5281/zenodo.XXXXXXX

This is the notebook that produced the published results. Run it top to bottom in Google Colab (random seed 42 throughout).

| Part | Content | Manuscript |
|---|---|---|
| A | Wells, yield classes, pseudo-absences (RF), 10,000 background points (MaxEnt), predictor extraction | Section 2.2–2.3, Table 2, Fig. 3 |
| B | Collinearity (Spearman, VIF) and spatial structure of wells | Section 3.2.1, Fig. 6 |
| C | Random Forest: stratified 70/30 split, 60-combination random search | Section 2.4.2, Table 3 |
| D | MaxEnt (elapid) on the same training wells and the same test set | Section 2.4.1 |
| E | Test metrics with bootstrap 95% CI, paired ΔAUC, calibration, 15 km spatial block CV | Section 2.5, Table 4, Fig. 5 |
| F | Variable importance, response curves, GLDAS-storage test | Section 3.2, Figs. 7–8 |
| G | Supplementary five-class yield model (unweighted vs cost-sensitive; no resampling) | Section 2.4.3, Table 5 |
| H | Current GWPZ maps and district areas; yield of wells by mapped zone | Section 3.3, Figs. 9–10 |
| I | RF–MaxEnt agreement | Section 3.5, Fig. 11 |
| J | CMIP6 scenarios: delta-change rainfall, only rainfall varied, extrapolation check, area change, district sensitivity | Sections 2.6, 3.4, Figs. 12–15, Table 6, Appendices A–B |
| K | Workflow figure, software environment, Excel workbooks, GeoTIFFs | Fig. 4, Data availability |

**Inputs in `/content/`:** `GW_SSPs.csv` (RWB boreholes with X, Y in EPSG:32735, `yield`, `GWPZ`, `ID`), the 12 predictor rasters, the six CMIP6 ensemble annual-precipitation rasters and the WorldClim 1970–2000 annual-precipitation baseline (file names in Part 1).

**Interpretation.** Outputs are modeled groundwater potential (relative suitability for groundwater occurrence learned from drilled boreholes), not recharge or availability. In the scenarios only rainfall changes; the other eleven predictors stay at present conditions, so scenario maps are model responses to precipitation, not forecasts.

In [ ]:
!pip -q install geopandas rasterio openpyxl scikit-learn statsmodels elapid==1.0.4 shapely

## 1. Configuration

In [ ]:
CSV_PATH = "/content/GW_SSPs.csv"

# ---- current predictors (key = CSV column name) ----
USER_RASTERS = {
    "DEM":                  "/content/DEM_projected.tif",
    "Distance_River":       "/content/River_D.tif",
    "Drainage_Density":     "/content/Drainage.tif",
    "Geology":              "/content/Geology.tif",
    "Ground_water_storage": "/content/GroundW_S.tif",
    "LULC":                 "/content/LULC.tif",
    "NDVI":                 "/content/NDVI.tif",
    "Rainfall":             "/content/Rainfall.tif",          # CHIRPS v2.0 mean annual rainfall
    "Slope":                "/content/Slope.tif",
    "Soi_depth":            "/content/Soi_Dep.tif",
    "Soil_Texture":         "/content/Soil_T.tif",
}
OPTIONAL_RASTERS = {"TWI": "/content/TWI.tif"}       # 12th predictor
CATEGORICAL = ["Geology", "LULC", "Soil_Texture", "Soi_depth"]

# ---- future rainfall: World Bank CCKP CMIP6 multi-model ensemble, annual precipitation (mm) ----
FUTURE_RAINFALL = {
    ("SSP1-2.6", "2040-2059"): "/content/ssp126_2040.tif",
    ("SSP1-2.6", "2080-2099"): "/content/ssp126_2080.tif",
    ("SSP2-4.5", "2040-2059"): "/content/ssp245_2040.tif",
    ("SSP2-4.5", "2080-2099"): "/content/ssp245_2080.tif",
    ("SSP5-8.5", "2040-2059"): "/content/ssp585_2040.tif",
    ("SSP5-8.5", "2080-2099"): "/content/ssp585_2080.tif",
}
CLIMATE_BASELINE = "/content/wc2.1_bio12_baseline.tif"   # WorldClim v2.1, 1970–2000 annual precipitation

# ---- GWPZ classes on the predicted probability (identical for all models and scenarios) ----
CLASS_NAMES  = ["Very Low", "Low", "Moderate", "High", "Very High"]
CLASS_COLORS = ["#d7191c", "#fdae61", "#ffffbf", "#a6d96a", "#1a9641"]
PROB_BREAKS  = [0.2, 0.4, 0.6, 0.8]

# ---- yield classes (m³/h), Table 2 ----
YIELD_BREAKS = [0.9, 1.6, 2.1, 2.6]

# ---- pseudo-absences (RF) and background (MaxEnt) ----
ABSENCE_RATIO             = 1.0
ABSENCE_STRATEGY          = "area"
MIN_DIST_TO_PRESENCE      = 1000
MIN_DIST_BETWEEN_ABSENCES = 500
OVERSAMPLE                = 4
WATER_CODES               = []
N_BACKGROUND              = 10000

# ---- partition, tuning, validation ----
TEST_SIZE   = 0.30
SEED        = 42
N_SEARCH    = 60
N_BOOT      = 2000
BLOCK_SIZE  = 15000
N_SPATIAL_FOLDS = 5
PRED_RES    = 100
OUT_DIR     = "/content/outputs"

## 2. Imports and helpers

In [ ]:
import os, sys, glob, math, json, warnings, platform, itertools
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch, FancyBboxPatch, Circle
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
from rasterio import features as rfeatures
from shapely import contains_xy, prepare
from scipy import stats
from scipy.spatial import cKDTree
import sklearn
from sklearn.model_selection import (train_test_split, RandomizedSearchCV, StratifiedKFold, GroupKFold)
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             cohen_kappa_score, roc_auc_score, roc_curve, brier_score_loss,
                             confusion_matrix, classification_report)
from sklearn.calibration import calibration_curve
from sklearn.base import clone
import elapid
from statsmodels.stats.outliers_influence import variance_inflation_factor
import joblib
warnings.filterwarnings("ignore")
os.makedirs(OUT_DIR, exist_ok=True)
FIG = f"{OUT_DIR}/figures"; os.makedirs(FIG, exist_ok=True)
rng = np.random.default_rng(SEED)
RESULTS = {}                                         # every table for the Excel workbook
plt.rcParams.update({"pdf.fonttype": 42, "savefig.facecolor": "white"})

def _union(g): return g.union_all() if hasattr(g, "union_all") else g.unary_union
def savefig(name):
    plt.savefig(f"{FIG}/{name}.png", dpi=600, bbox_inches="tight")
    plt.savefig(f"{FIG}/{name}.pdf", bbox_inches="tight")
def classify(p): return np.digitize(p, PROB_BREAKS) + 1          # 1..5
print("✅ ready | sklearn", sklearn.__version__, "| elapid", elapid.__version__)

# Part A – Data
## 3. Wells, yield classes and thresholds (Table 2)

In [ ]:
df = pd.read_csv(CSV_PATH); df.columns = df.columns.str.strip()
n0 = len(df)
dups = df[df.duplicated(subset=["X", "Y"], keep="first")]
df = df.dropna(subset=["X", "Y"]).drop_duplicates(subset=["X", "Y"]).reset_index(drop=True)
print(f"✅ {len(df)} wells ({n0-len(df)} duplicate coordinates removed: IDs {dups['ID'].tolist()})")

df["Yield_Class"] = np.digitize(df["yield"], YIELD_BREAKS) + 1
agree = (df["Yield_Class"] == df["GWPZ"]).mean() * 100
print(f"Thresholds reproduce the CSV GWPZ column for {agree:.1f}% of wells")

thr = (df.groupby("Yield_Class")["yield"]
         .agg(n_wells="count", min_yield="min", median_yield="median", max_yield="max")
         .reindex(range(1, 6)).reset_index())
thr.insert(1, "Class", CLASS_NAMES)
thr.insert(2, "Yield_rule_m3h", [f"< {YIELD_BREAKS[0]}"] +
           [f"{YIELD_BREAKS[i]} – <{YIELD_BREAKS[i+1]}" for i in range(3)] + [f"≥ {YIELD_BREAKS[3]}"])
thr["Percent"] = (thr.n_wells / thr.n_wells.sum() * 100).round(1)
print(thr.to_string(index=False))
RESULTS["T_Yield_Class_Thresholds"] = thr

## 4. Predictor rasters – check and metadata (Table 1)

In [ ]:
for k, p in OPTIONAL_RASTERS.items():
    if os.path.exists(p): USER_RASTERS[k] = p
    else: print(f"ℹ️ optional layer {k} not found ({p}) – model uses {len(USER_RASTERS)} predictors")
missing = [p for p in USER_RASTERS.values() if not os.path.exists(p)]
if missing: raise FileNotFoundError(f"Upload these rasters to /content: {missing}")
FEATURES = list(USER_RASTERS)
CATEGORICAL = [c for c in CATEGORICAL if c in FEATURES]
CONTINUOUS = [c for c in FEATURES if c not in CATEGORICAL]

meta = []
for col, path in USER_RASTERS.items():
    with rasterio.open(path) as src:
        a = src.read(1, masked=True)
        meta.append({"Layer": col, "File": Path(path).name, "Type": "categorical" if col in CATEGORICAL else "continuous",
                     "CRS": str(src.crs), "Pixel_size": f"{src.res[0]:.1f} x {src.res[1]:.1f}",
                     "Rows x Cols": f"{src.height} x {src.width}", "NoData": src.nodata,
                     "Min": float(a.min()), "Max": float(a.max()),
                     "Resampling_to_grid": "nearest" if col in CATEGORICAL else "bilinear"})
meta = pd.DataFrame(meta); print(meta.to_string(index=False))
RESULTS["T_Predictor_Metadata"] = meta

## 5. Boundaries (GADM 4.1)

In [ ]:
rwa_prov = gpd.read_file("https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_RWA_1.json")
rwa_dist = gpd.read_file("https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_RWA_2.json")
prov_col = next(c for c in rwa_prov.columns
                if rwa_prov[c].dtype == object and any("Eastern" in str(v) for v in rwa_prov[c]))
eastern_prov = rwa_prov[rwa_prov[prov_col].str.contains("Eastern", na=False)].copy()
eastern_dist = rwa_dist[rwa_dist["GID_1"] == eastern_prov["GID_1"].values[0]].copy()
dist_name_col = next(
    (c for c in eastern_dist.columns
     if any(v in eastern_dist[c].astype(str).tolist()
            for v in ["Nyagatare", "Gatsibo", "Kayonza", "Ngoma", "Kirehe", "Bugesera", "Rwamagana"])),
    [c for c in eastern_dist.columns if "NAME" in c][-1])
DST_CRS = "EPSG:32735"
eastern_prov_utm = eastern_prov.to_crs(DST_CRS)
eastern_dist_utm = eastern_dist.to_crs(DST_CRS)
print(f"✅ Districts: {eastern_dist[dist_name_col].tolist()} | {DST_CRS}")

## 6. Presence points

In [ ]:
prov_geom = _union(eastern_prov_utm.geometry)
dist_gdf = (eastern_dist_utm[[dist_name_col, "geometry"]]
            .rename(columns={dist_name_col: "District"}).reset_index(drop=True))
DISTRICTS = dist_gdf.District.tolist()

pres = gpd.GeoDataFrame(df.copy(), geometry=gpd.points_from_xy(df["X"], df["Y"]), crs=DST_CRS)
inside = pres.within(prov_geom.buffer(1000))
if (~inside).any(): print(f"⚠️ {(~inside).sum()} wells outside the province removed")
pres = pres[inside].reset_index(drop=True)
pres = gpd.sjoin_nearest(pres, dist_gdf, how="left").drop(columns="index_right")
pres = pres[~pres.index.duplicated()]
pres["Class"] = 1
RESULTS["T_Wells_by_District_Class"] = pd.crosstab(pres.District, pres.Yield_Class.map(dict(enumerate(CLASS_NAMES, 1))), margins=True).reset_index()
print(RESULTS["T_Wells_by_District_Class"])

## 7. Pseudo-absences (RF) and background points (MaxEnt)

In [ ]:
def quotas(n, weights):
    w = np.asarray(weights, float); w = w / w.sum()
    raw = w * n; q = np.floor(raw).astype(int)
    q[np.argsort(-(raw - q))[: n - q.sum()]] += 1
    return q

def sample_polygon(poly, n, rng, accepted, min_d, max_rounds=300):
    out = []
    if n <= 0 or poly.is_empty: return out
    prepare(poly); minx, miny, maxx, maxy = poly.bounds
    for _ in range(max_rounds):
        xs = rng.uniform(minx, maxx, max(200, n * 20)); ys = rng.uniform(miny, maxy, max(200, n * 20))
        m = contains_xy(poly, xs, ys)
        for x, y in zip(xs[m], ys[m]):
            if accepted and min_d > 0:
                a = np.asarray(accepted)
                if np.min(np.hypot(a[:, 0] - x, a[:, 1] - y)) < min_d: continue
            accepted.append((x, y)); out.append((x, y))
            if len(out) >= n: return out
    return out

n_target = int(round(len(pres) * ABSENCE_RATIO))
weights = (dist_gdf.area.values if ABSENCE_STRATEGY == "area"
           else dist_gdf.District.map(pres.District.value_counts()).fillna(0).values)
dist_gdf["w"] = weights
exclusion = _union(pres.buffer(MIN_DIST_TO_PRESENCE))
accepted, rows = [], []
for (_, d), q in zip(dist_gdf.iterrows(), quotas(n_target * OVERSAMPLE, weights)):
    rows += [{"District": d.District, "X": x, "Y": y}
             for x, y in sample_polygon(d.geometry.difference(exclusion), int(q), rng, accepted, MIN_DIST_BETWEEN_ABSENCES)]
cand = pd.DataFrame(rows)
cand = gpd.GeoDataFrame(cand, geometry=gpd.points_from_xy(cand.X, cand.Y), crs=DST_CRS); cand["Class"] = 0

# MaxEnt background: random over the whole province
bg_xy = sample_polygon(prov_geom, N_BACKGROUND, np.random.default_rng(SEED + 1), [], 0)
bg = gpd.GeoDataFrame(pd.DataFrame(bg_xy, columns=["X", "Y"]),
                      geometry=gpd.points_from_xy(*zip(*bg_xy)), crs=DST_CRS)
print(f"✅ {len(cand)} absence candidates, {len(bg)} background points")

## 8. Extract predictors (same rasters for wells, pseudo-absences and background)

In [ ]:
def extract(gdf, layers):
    out = pd.DataFrame(index=gdf.index)
    for col, path in layers.items():
        with rasterio.open(path) as src:
            g = gdf.to_crs(src.crs or DST_CRS)
            v = np.array([s.filled(np.nan)[0] for s in
                          src.sample([(p.x, p.y) for p in g.geometry], masked=True)], float)
        out[col] = v
    return out

pres["Source"], cand["Source"] = "Field well", "Generated absence"
keep = ["Class", "Source", "District", "X", "Y", "yield", "Yield_Class", "geometry"]
pts = gpd.GeoDataFrame(pd.concat([pres[keep], cand.reindex(columns=keep)], ignore_index=True),
                       geometry="geometry", crs=DST_CRS)
pts = pd.concat([pts, extract(pts, USER_RASTERS)], axis=1)
bg = pd.concat([bg, extract(bg, USER_RASTERS)], axis=1).dropna(subset=FEATURES).reset_index(drop=True)
ll = pts.to_crs(4326); pts["Lon"], pts["Lat"] = ll.geometry.x.values, ll.geometry.y.values

chk = []
w = pts[pts.Class == 1].reset_index(drop=True)
for f in FEATURES:
    if f in pres.columns:
        a, b = pd.to_numeric(pres[f], errors="coerce").values, w[f].values
        ok = np.isfinite(a) & np.isfinite(b)
        chk.append({"Layer": f, "Pearson_r": round(np.corrcoef(a[ok], b[ok])[0, 1], 3) if ok.sum() > 2 else np.nan,
                    "Exact_match_%": round(100 * np.mean(np.isclose(a[ok], b[ok], rtol=1e-3)), 1)})
chk = pd.DataFrame(chk); print(chk.to_string(index=False))
RESULTS["T_CSV_vs_Raster_Check"] = chk

valid = pts[FEATURES].notna().all(axis=1)
P = pts[(pts.Class == 1) & valid].copy()
A = pts[(pts.Class == 0) & valid].copy()
if WATER_CODES and "LULC" in FEATURES: A = A[~A.LULC.isin(WATER_CODES)]
n_target = int(round(len(P) * ABSENCE_RATIO)); sel = []
for dname, q in zip(dist_gdf.District, quotas(n_target, dist_gdf.w.values)):
    sel += A[A.District == dname].index[:q].tolist()
sel += A.drop(index=sel).index[: max(0, n_target - len(sel))].tolist()
data = gpd.GeoDataFrame(pd.concat([P, A.loc[sel]]).reset_index(drop=True), geometry="geometry", crs=DST_CRS)
data.insert(0, "Sample_ID", [f"{'P' if c else 'A'}{i+1:04d}" for i, c in enumerate(data.Class)])
data["Class_Label"] = np.where(data.Class == 1, "Potential", "Non-potential")
print(f"✅ dataset: {data.Class.value_counts().to_dict()}  | background: {len(bg)}")

## 9. Fig. 3 – wells and pseudo-absences; wells by yield class

In [ ]:
def base_map(ax):
    eastern_dist_utm.boundary.plot(ax=ax, color="grey", lw=0.6)
    eastern_prov_utm.boundary.plot(ax=ax, color="black", lw=1.2)
    for _, r in dist_gdf.iterrows():
        p = r.geometry.representative_point(); ax.annotate(r.District, (p.x, p.y), ha="center", fontsize=8, alpha=.8)
    ax.set_xticks([]); ax.set_yticks([])

fig, ax = plt.subplots(1, 2, figsize=(14, 9))
base_map(ax[0]); data[data.Class == 1].plot(ax=ax[0], color="#2c7bb6", markersize=18, edgecolor="k", lw=.3,
                                           label=f"Wells (n={int(data.Class.sum())})")
data[data.Class == 0].plot(ax=ax[0], color="#bdbdbd", marker="x", markersize=14,
                           label=f"Pseudo-absences (n={int((data.Class==0).sum())})")
ax[0].legend(loc="lower left"); ax[0].set_title("(a) Presence data (MaxEnt) and pseudo-absences (RF)")
base_map(ax[1])
for k in range(1, 6):
    s = data[(data.Class == 1) & (data.Yield_Class == k)]
    s.plot(ax=ax[1], color=CLASS_COLORS[k-1], markersize=26, edgecolor="k", lw=.4,
           label=f"{k} {CLASS_NAMES[k-1]} ({thr.Yield_rule_m3h[k-1]} m³/h), n={len(s)}")
ax[1].legend(loc="lower left", fontsize=8, title="Yield class"); ax[1].set_title("(b) Wells by yield class")
plt.tight_layout(); savefig("Fig03_wells_classes"); plt.show()

# Part B – Diagnostics
## 10. Collinearity (Spearman, VIF; Fig. 6) and spatial structure of wells

In [ ]:
corr = data[FEATURES].corr(method="spearman")
fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(FEATURES))); ax.set_xticklabels(FEATURES, rotation=90)
ax.set_yticks(range(len(FEATURES))); ax.set_yticklabels(FEATURES)
for i in range(len(FEATURES)):
    for j in range(len(FEATURES)):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=7)
plt.colorbar(im, label="Spearman ρ"); plt.title("Predictor correlation (wells + pseudo-absences)")
savefig("Fig06_correlation"); plt.show()

Xv = data[CONTINUOUS].assign(const=1.0).values
vif = pd.DataFrame({"Predictor": CONTINUOUS,
                    "VIF": [variance_inflation_factor(Xv, i) for i in range(len(CONTINUOUS))]}).round(2)
vif["Assessment"] = np.where(vif.VIF > 10, "severe", np.where(vif.VIF > 5, "moderate", "acceptable"))
print(vif.to_string(index=False))
pairs = [(a, b, round(corr.loc[a, b], 2)) for i, a in enumerate(FEATURES) for b in FEATURES[i+1:] if abs(corr.loc[a, b]) >= 0.7]
print("|ρ| ≥ 0.7:", pairs or "none")
RESULTS["T_Collinearity_VIF"] = vif
RESULTS["T_Spearman_Matrix"] = corr.round(3).reset_index().rename(columns={"index": "Predictor"})

xy = data.loc[data.Class == 1, ["X", "Y"]].values
dnn = cKDTree(xy).query(xy, k=2)[0][:, 1]
R_ce = dnn.mean() / (0.5 / np.sqrt(len(xy) / prov_geom.area))
nn = pd.DataFrame({"Statistic": ["n wells", "NN distance min (m)", "NN distance median (m)", "NN distance mean (m)",
                                 "Wells with neighbour < 1 km (%)", "Clark–Evans R (<1 = clustered)", "Spatial CV block size (m)"],
                   "Value": [len(xy), dnn.min().round(0), np.median(dnn).round(0), dnn.mean().round(0),
                             round((dnn < 1000).mean() * 100, 1), round(R_ce, 3), BLOCK_SIZE]})
print(nn.to_string(index=False)); RESULTS["T_Well_Spatial_Structure"] = nn

# Part C – Random Forest groundwater-potential model
## 11. Stratified 70/30 split and 15 km spatial blocks

In [ ]:
X, y = data[FEATURES], data["Class"].values
idx_tr, idx_te = train_test_split(data.index, test_size=TEST_SIZE, stratify=y, random_state=SEED)
data["Split"] = "Train"; data.loc[idx_te, "Split"] = "Test"
X_tr, X_te, y_tr, y_te = X.loc[idx_tr], X.loc[idx_te], y[idx_tr], y[idx_te]
blocks = (np.floor(data.X / BLOCK_SIZE).astype(int).astype(str) + "_" + np.floor(data.Y / BLOCK_SIZE).astype(int).astype(str)).values
split_tab = pd.crosstab(data.Class_Label, data.Split, margins=True)
print(split_tab); print(f"{len(np.unique(blocks))} spatial blocks of {BLOCK_SIZE/1000:.0f} km")
RESULTS["T_Train_Test_Counts"] = split_tab.reset_index()

## 12. RF tuning (RandomizedSearchCV, 60 combinations, 5-fold stratified CV; Table 3)

In [ ]:
param_dist = {"n_estimators": [200, 300, 400, 500, 600, 800], "max_depth": [None, 5, 10, 20, 30],
              "min_samples_split": [2, 3, 5, 10], "min_samples_leaf": [1, 2, 3, 5],
              "max_features": ["sqrt", "log2", 0.5], "class_weight": [None, "balanced", "balanced_subsample"]}
search = RandomizedSearchCV(RandomForestClassifier(random_state=SEED, n_jobs=-1, bootstrap=True, oob_score=True),
                            param_dist, n_iter=N_SEARCH, scoring="roc_auc",
                            cv=StratifiedKFold(5, shuffle=True, random_state=SEED), random_state=SEED, n_jobs=-1)
search.fit(X_tr, y_tr)
rf = search.best_estimator_
rf_cfg = pd.DataFrame({"Setting": ["Software", "Algorithm", "Tuning", "Scoring", "Random seed", "Predictors"] +
                       [f"best {k}" for k in search.best_params_] + ["CV AUC (train, 5-fold)", "OOB accuracy"],
                       "Value": [f"scikit-learn {sklearn.__version__}", "RandomForestClassifier (bootstrap=True)",
                                 f"RandomizedSearchCV, {N_SEARCH} combinations", "ROC-AUC", SEED, ", ".join(FEATURES)] +
                                [str(v) for v in search.best_params_.values()] +
                                [round(search.best_score_, 4), round(rf.oob_score_, 4)]})
print(rf_cfg.to_string(index=False)); RESULTS["T_RF_Configuration"] = rf_cfg
RESULTS["T_RF_SearchSpace"] = pd.DataFrame([(k, str(v)) for k, v in param_dist.items()], columns=["Hyperparameter", "Search space"])

# Part D – MaxEnt (elapid), same training wells and same test set
Training = the 106 training wells + 10,000 background points. Test = the same 46 wells and 46 pseudo-absences as RF.

*Categorical predictors.* Geology, LULC, soil texture and soil depth are integer class codes. With a DataFrame as input, elapid 1.0.4 does not apply the `categorical=` argument (predictions are identical with and without it), so in the published model these predictors entered MaxEnt as numeric codes, as in RF. The call below states this explicitly and gives identical results.

In [ ]:
ME_SETTINGS = dict(feature_types=["linear", "quadratic", "hinge", "product"], beta_multiplier=1.0,
                   transform="cloglog", clamp=True, tau=0.5, n_hinge_features=10, random_state=SEED)
def fit_maxent(pres_X):
    xm = pd.concat([pres_X[FEATURES], bg[FEATURES]], ignore_index=True)
    ym = np.r_[np.ones(len(pres_X)), np.zeros(len(bg))]
    m = elapid.MaxentModel(**ME_SETTINGS); m.fit(xm, ym); return m

me = fit_maxent(X_tr[y_tr == 1])
me_pred = lambda Xd: np.asarray(me.predict(Xd[FEATURES])).ravel()
auc_bg_test = roc_auc_score(np.r_[np.ones((y_te == 1).sum()), np.zeros(len(bg))],
                            np.r_[me_pred(X_te[y_te == 1]), me_pred(bg)])
me_cfg = pd.DataFrame({"Setting": ["Software", "Feature classes", "Regularisation multiplier", "Output", "Clamping",
                                   "Prevalence (tau)", "Categorical predictors", "Background points", "Training presences",
                                   "Test presences", "Replicate type", "Test AUC (presence vs background)"],
                       "Value": [f"elapid {elapid.__version__} (Python MaxEnt)", "linear, quadratic, hinge, product", 1.0,
                                 "cloglog", "yes", 0.5, "numeric class codes (as in RF)", len(bg), int((y_tr == 1).sum()),
                                 int((y_te == 1).sum()), f"single 70/30 split (seed {SEED}) + {N_SPATIAL_FOLDS}-fold spatial block CV",
                                 round(auc_bg_test, 4)]})
print(me_cfg.to_string(index=False)); RESULTS["T_MaxEnt_Configuration"] = me_cfg

# Part E – Validation
## 13. Test-set metrics with bootstrap 95% CI, paired ΔAUC and calibration (Table 4, Fig. 5)

In [ ]:
p_rf_tr = rf.oob_decision_function_[:, 1]                 # out-of-bag → unbiased threshold
p_me_tr = me_pred(X_tr)
p_rf, p_me = rf.predict_proba(X_te)[:, 1], me_pred(X_te)

def best_thr(yy, pp):                                      # threshold maximising TSS
    f, t, th = roc_curve(yy, pp); return th[np.argmax(t - f)]
THR = {"RF": best_thr(y_tr, p_rf_tr), "MaxEnt": best_thr(y_tr, p_me_tr)}

def mets(yy, pp, thr):
    pr = (pp >= thr).astype(int); sens = recall_score(yy, pr); spec = recall_score(yy, pr, pos_label=0)
    return {"AUC": roc_auc_score(yy, pp), "Accuracy": accuracy_score(yy, pr), "Precision": precision_score(yy, pr, zero_division=0),
            "Recall (sensitivity)": sens, "Specificity": spec, "F1": f1_score(yy, pr), "Cohen kappa": cohen_kappa_score(yy, pr),
            "TSS": sens + spec - 1, "Brier score": brier_score_loss(yy, np.clip(pp, 0, 1))}

def boot(yy, pp, thr, n=N_BOOT, seed=SEED):
    r = np.random.default_rng(seed); out = []
    for _ in range(n):
        i = r.integers(0, len(yy), len(yy))
        if len(np.unique(yy[i])) < 2: continue
        out.append(mets(yy[i], pp[i], thr))
    return pd.DataFrame(out)

rows = []
for name, pp in [("RF", p_rf), ("MaxEnt", p_me)]:
    pt, bs = mets(y_te, pp, THR[name]), boot(y_te, pp, THR[name])
    for k, v in pt.items():
        rows.append({"Model": name, "Metric": k, "Value": round(v, 3),
                     "CI95_low": round(bs[k].quantile(.025), 3), "CI95_high": round(bs[k].quantile(.975), 3)})
    rows.append({"Model": name, "Metric": "Threshold (max TSS on training)", "Value": round(THR[name], 3)})
test_metrics = pd.DataFrame(rows)
print(test_metrics.pivot(index="Metric", columns="Model", values="Value"))

r = np.random.default_rng(SEED); d_auc = []
for _ in range(N_BOOT):
    i = r.integers(0, len(y_te), len(y_te))
    if len(np.unique(y_te[i])) < 2: continue
    d_auc.append(roc_auc_score(y_te[i], p_rf[i]) - roc_auc_score(y_te[i], p_me[i]))
d_auc = np.array(d_auc)
p_val = 2 * min((d_auc <= 0).mean(), (d_auc >= 0).mean())
cmp_tab = pd.DataFrame({"Statistic": ["ΔAUC (RF − MaxEnt)", "95% CI low", "95% CI high", "two-sided bootstrap p"],
                        "Value": [round(roc_auc_score(y_te, p_rf) - roc_auc_score(y_te, p_me), 4),
                                  round(np.quantile(d_auc, .025), 4), round(np.quantile(d_auc, .975), 4), round(p_val, 4)]})
print(cmp_tab.to_string(index=False))
RESULTS["T_Test_Metrics_CI"] = test_metrics; RESULTS["T_RF_vs_MaxEnt_DeltaAUC"] = cmp_tab

# Fig. 5 – ROC, calibration (8 quantile bins) and confusion matrices, 2 × 2 layout as in the article
plt.rcParams.update({"font.size": 7, "axes.titlesize": 7.5, "axes.labelsize": 7, "xtick.labelsize": 6.5, "ytick.labelsize": 6.5})
fig, ax = plt.subplots(2, 2, figsize=(7.4, 6.8))
for name, pp, c in [("RF", p_rf, "#1b9e77"), ("MaxEnt", p_me, "#d95f02")]:
    f, t, _ = roc_curve(y_te, pp)
    lo, hi = test_metrics.query("Model==@name and Metric=='AUC'")[["CI95_low", "CI95_high"]].values[0]
    ax[0, 0].plot(f, t, c=c, lw=1.4, label=f"{name}: AUC = {roc_auc_score(y_te, pp):.3f}\n(95% CI {lo:.3f}–{hi:.3f})")
    fr, mp = calibration_curve(y_te, np.clip(pp, 0, 1), n_bins=8, strategy="quantile")
    ax[0, 1].plot(mp, fr, "o-", c=c, ms=3.5, lw=1.2, label=f"{name} (Brier = {brier_score_loss(y_te, np.clip(pp,0,1)):.3f})")
for a in (ax[0, 0], ax[0, 1]):
    a.plot([0, 1], [0, 1], ls="--", c="0.4", lw=.8); a.set_xlim(-.02, 1.02); a.set_ylim(-.02, 1.02); a.set_aspect("equal"); a.grid(alpha=.3, lw=.4)
ax[0, 0].legend(loc="lower right", fontsize=6.3); ax[0, 1].legend(loc="upper left", fontsize=6.3)
ax[0, 0].set(xlabel="False positive rate (1 − specificity)", ylabel="True positive rate (sensitivity)", title="(a) ROC curves, test set")
ax[0, 1].set(xlabel="Mean predicted value", ylabel="Observed fraction of wells", title="(b) Calibration (reliability)")
for k, (name, pp) in enumerate([("RF", p_rf), ("MaxEnt", p_me)]):
    cm = confusion_matrix(y_te, (pp >= THR[name]).astype(int)); a = ax[1, k]; rp = cm / cm.sum(1, keepdims=True)
    a.imshow(rp, cmap="Blues", vmin=0, vmax=1)
    for i in range(2):
        for j in range(2): a.text(j, i, f"{cm[i,j]}\n({rp[i,j]:.0%})", ha="center", va="center", fontsize=8, color="white" if rp[i, j] > .55 else "black")
    a.set_xticks([0, 1], ["Pseudo-absence", "Well"]); a.set_yticks([0, 1], [f"Pseudo-abs.\n(n = {cm[0].sum()})", f"Well\n(n = {cm[1].sum()})"] if k == 0 else ["", ""])
    a.set_xlabel("Predicted"); a.set_ylabel("Observed" if k == 0 else "")
    a.set_title(f"({'cd'[k]}) {name} confusion matrix (threshold {THR[name]:.3f})")
for a in ax.ravel(): a.title.set_fontsize(8); a.title.set_ha("left"); a.title.set_position((0, 1))
plt.tight_layout(w_pad=2.5, h_pad=1.5); savefig("Fig05_ROC_calibration_confusion"); plt.show(); plt.rcdefaults(); plt.rcParams.update({"pdf.fonttype": 42, "savefig.facecolor": "white"})

## 14. Spatial block cross-validation (15 km blocks, 5 folds)

In [ ]:
gkf = GroupKFold(N_SPATIAL_FOLDS); cv_rows = []
oof_rf = np.full(len(data), np.nan); oof_me = np.full(len(data), np.nan)
for k, (tr, te) in enumerate(gkf.split(X, y, groups=blocks), 1):
    m = clone(rf).set_params(oob_score=False).fit(X.iloc[tr], y[tr])
    oof_rf[te] = m.predict_proba(X.iloc[te])[:, 1]
    mm = fit_maxent(X.iloc[tr][y[tr] == 1]); oof_me[te] = np.asarray(mm.predict(X.iloc[te][FEATURES])).ravel()
    if len(np.unique(y[te])) == 2:
        for name, pp in [("RF", oof_rf[te]), ("MaxEnt", oof_me[te])]:
            met = mets(y[te], pp, THR[name]); cv_rows.append({"Fold": k, "Model": name, "n_test": len(te), **met})
cvdf = pd.DataFrame(cv_rows)
cv_sum = cvdf.groupby("Model")[["AUC", "Accuracy", "Cohen kappa", "TSS", "F1"]].agg(["mean", "std"]).round(3)
print(cv_sum)
RESULTS["T_SpatialCV_Folds"] = cvdf.round(4)
RESULTS["T_SpatialCV_Summary"] = cv_sum.reset_index().set_axis(["Model"] + [f"{a}_{b}" for a, b in cv_sum.columns], axis=1)
data["OOF_Spatial_RF"], data["OOF_Spatial_MaxEnt"] = oof_rf, oof_me

# Part F – Variable importance (Fig. 7), response curves (Fig. 8) and GLDAS-storage test

In [ ]:
def perm_importance(predict, Xd, yy, n_rep=50, seed=SEED):
    r = np.random.default_rng(seed); base = roc_auc_score(yy, predict(Xd)); out = []
    for f in FEATURES:
        drops = []
        for _ in range(n_rep):
            Xp = Xd.copy(); Xp[f] = r.permutation(Xp[f].values); drops.append(base - roc_auc_score(yy, predict(Xp)))
        out.append((f, np.mean(drops), np.std(drops)))
    return pd.DataFrame(out, columns=["Feature", "mean", "sd"])

pi_rf = perm_importance(lambda d: rf.predict_proba(d)[:, 1], X_te, y_te)
pi_me = perm_importance(me_pred, X_te, y_te)
imp = pd.DataFrame({"Feature": FEATURES, "RF_MDI": rf.feature_importances_})
imp = imp.merge(pi_rf.rename(columns={"mean": "RF_Perm_AUCdrop", "sd": "RF_Perm_SD"}), on="Feature")
imp = imp.merge(pi_me.rename(columns={"mean": "MaxEnt_Perm_AUCdrop", "sd": "MaxEnt_Perm_SD"}), on="Feature")
for c in ["RF_MDI", "RF_Perm_AUCdrop", "MaxEnt_Perm_AUCdrop"]:
    imp[c.replace("_AUCdrop", "") + "_Rank"] = imp[c].rank(ascending=False).astype(int)
imp["Max_|rho|_with_other"] = [round(float(corr.loc[f].drop(f).abs().max()), 2) for f in imp.Feature]
imp = imp.sort_values("RF_Perm_AUCdrop", ascending=False).round(4)
print(imp.to_string(index=False)); RESULTS["T_Variable_Importance"] = imp

o = imp.sort_values("RF_MDI")
fig, ax = plt.subplots(1, 3, figsize=(19, 6))
ax[0].barh(o.Feature, o.RF_MDI, color="#4c72b0"); ax[0].set_title("(a) RF – Mean Decrease in Impurity\n(training data, no uncertainty)")
o = imp.sort_values("RF_Perm_AUCdrop")
ax[1].barh(o.Feature, o.RF_Perm_AUCdrop, xerr=o.RF_Perm_SD, color="#55a868", capsize=3)
ax[1].set_title("(b) RF – permutation importance\n(AUC drop on test set, mean ± SD, 50 repeats)")
o = imp.sort_values("MaxEnt_Perm_AUCdrop")
ax[2].barh(o.Feature, o.MaxEnt_Perm_AUCdrop, xerr=o.MaxEnt_Perm_SD, color="#dd8452", capsize=3)
ax[2].set_title("(c) MaxEnt – permutation importance\n(AUC drop on test set, mean ± SD)")
for a in ax[1:]: a.axvline(0, c="k", lw=.8)
plt.tight_layout(); savefig("Fig07_variable_importance"); plt.show()

# Response (partial-dependence) curves computed from the fitted models (Fig. 8)
def pdp(predict, f, Xref, n=30):
    Xs = Xref.sample(min(400, len(Xref)), random_state=SEED)
    grid = np.sort(Xref[f].unique()) if f in CATEGORICAL else np.unique(np.quantile(Xref[f], np.linspace(.02, .98, n)))
    vals = []
    for g in grid:
        Xg = Xs.copy(); Xg[f] = g; vals.append(np.mean(predict(Xg)))
    return grid, np.array(vals)

ncol = 4; nrow = math.ceil(len(FEATURES) / ncol)
fig, axs = plt.subplots(nrow, ncol, figsize=(18, 4 * nrow)); axs = axs.ravel()
for a, f in zip(axs, imp.Feature):
    for name, fn, c in [("RF", lambda d: rf.predict_proba(d)[:, 1], "#1b9e77"), ("MaxEnt", me_pred, "#d95f02")]:
        g, v = pdp(fn, f, X_tr)
        a.plot(g, v, "o" if f in CATEGORICAL else "-", c=c, label=name, lw=2)
    a.set_title(f); a.set_ylim(0, 1); a.set_ylabel("Predicted GWP")
axs[0].legend()
for a in axs[len(FEATURES):]: a.axis("off")
plt.tight_layout(); savefig("Fig08_response_curves"); plt.show()

# GLDAS groundwater-storage circularity test: refit without GWS
if "Ground_water_storage" in FEATURES:
    F2 = [f for f in FEATURES if f != "Ground_water_storage"]
    rf2 = RandomForestClassifier(**{**rf.get_params(), "oob_score": False}).fit(X_tr[F2], y_tr)
    oof2 = np.full(len(data), np.nan)
    for tr, te in gkf.split(X, y, groups=blocks):
        oof2[te] = clone(rf2).fit(X.iloc[tr][F2], y[tr]).predict_proba(X.iloc[te][F2])[:, 1]
    gws_tab = pd.DataFrame({"Model": ["RF with GWS", "RF without GWS"],
        "Test AUC": [roc_auc_score(y_te, p_rf), roc_auc_score(y_te, rf2.predict_proba(X_te[F2])[:, 1])],
        "Spatial-CV AUC (pooled OOF)": [roc_auc_score(y, oof_rf), roc_auc_score(y, oof2)]}).round(4)
    print(gws_tab.to_string(index=False)); RESULTS["T_GWS_Circularity_Test"] = gws_tab

# Part G – Supplementary five-class yield model (Section 2.4.3, Table 5)
The 152 wells are split separately into 70% training and 30% test wells, stratified by yield class (seed 42); 46 test wells.
Two treatments of class imbalance are compared on the same test wells: a standard RF and a cost-sensitive RF (`class_weight='balanced'`). No resampling (SMOTE) is used.

In [ ]:
W = data[data.Class == 1].reset_index(drop=True)
Xw, yw = W[FEATURES], W["Yield_Class"].values
iw_tr, iw_te = train_test_split(W.index, test_size=TEST_SIZE, stratify=yw, random_state=SEED)
cnt = pd.DataFrame({"Class": CLASS_NAMES,
                    "Total": [int((yw == k).sum()) for k in range(1, 6)],
                    "Train": [int((yw[iw_tr] == k).sum()) for k in range(1, 6)],
                    "Test": [int((yw[iw_te] == k).sum()) for k in range(1, 6)]})
print(cnt.to_string(index=False))

yparams = {"rf__n_estimators": [300, 500, 800], "rf__max_depth": [None, 5, 10, 20],
           "rf__min_samples_leaf": [1, 2, 3, 5], "rf__max_features": ["sqrt", "log2", 0.5]}
def fit_yield(cw):
    s = RandomizedSearchCV(Pipeline([("rf", RandomForestClassifier(random_state=SEED, n_jobs=-1, class_weight=cw))]),
                           yparams, n_iter=30, scoring="f1_macro",
                           cv=StratifiedKFold(3, shuffle=True, random_state=SEED), random_state=SEED, n_jobs=-1)
    return s.fit(Xw.loc[iw_tr], yw[iw_tr]).best_estimator_

def ymets(yy, pp):
    pr = pp.argmax(1) + 1
    return {"Accuracy": accuracy_score(yy, pr), "Within-one-class accuracy": np.mean(np.abs(pr - yy) <= 1),
            "Macro F1": f1_score(yy, pr, average="macro", zero_division=0),
            "Cohen kappa": cohen_kappa_score(yy, pr), "Quadratic weighted kappa": cohen_kappa_score(yy, pr, weights="quadratic"),
            "Macro AUC (OvR)": roc_auc_score(yy, pp, multi_class="ovr", average="macro", labels=[1, 2, 3, 4, 5]),
            "Weighted AUC (OvR)": roc_auc_score(yy, pp, multi_class="ovr", average="weighted", labels=[1, 2, 3, 4, 5])}

models_y = {"Unweighted": fit_yield(None), "Cost-sensitive (class_weight=balanced)": fit_yield("balanced")}
yte = yw[iw_te]; abl = []
for name, m in models_y.items():
    pp = m.predict_proba(Xw.loc[iw_te]); pt = ymets(yte, pp)
    r = np.random.default_rng(SEED); bs = []
    for _ in range(N_BOOT):
        i = r.integers(0, len(yte), len(yte))
        if len(np.unique(yte[i])) == 5: bs.append(ymets(yte[i], pp[i]))
    bs = pd.DataFrame(bs)
    for k, v in pt.items():
        abl.append({"Model": name, "Metric": k, "Value": round(v, 3),
                    "CI95_low": round(bs[k].quantile(.025), 3), "CI95_high": round(bs[k].quantile(.975), 3)})
abl = pd.DataFrame(abl)
print(abl.pivot(index="Metric", columns="Model", values="Value"))
RESULTS["T_Yield_Class_Counts"] = cnt; RESULTS["T_Yield_RF_CI"] = abl

ym = models_y["Unweighted"]; ypp = ym.predict_proba(Xw.loc[iw_te]); ypr = ypp.argmax(1) + 1
rep = pd.DataFrame(classification_report(yte, ypr, labels=[1, 2, 3, 4, 5], target_names=CLASS_NAMES,
                                         output_dict=True, zero_division=0)).T.round(3)
rep.loc[CLASS_NAMES, "AUC"] = [round(roc_auc_score((yte == k).astype(int), ypp[:, k-1]), 3) for k in range(1, 6)]
print(rep); RESULTS["T_Yield_RF_Report"] = rep.reset_index().rename(columns={"index": "Class"})

# Part H – Current GWPZ maps (Fig. 9) and district areas (Fig. 10)
## 15. Prediction grid (all layers resampled directly to one 100 m grid)

In [ ]:
minx, miny, maxx, maxy = prov_geom.bounds
W_, H_ = int(math.ceil((maxx - minx) / PRED_RES)), int(math.ceil((maxy - miny) / PRED_RES))
GT = from_origin(minx, maxy, PRED_RES, PRED_RES)
MASK = rfeatures.geometry_mask([prov_geom], (H_, W_), GT, invert=True)
DIST_ID = rfeatures.rasterize([(g, i + 1) for i, g in enumerate(dist_gdf.geometry)], (H_, W_), transform=GT, fill=0)
PIX_KM2 = PRED_RES ** 2 / 1e6

def to_grid(path, categorical=False):
    out = np.full((H_, W_), np.nan, np.float32)
    with rasterio.open(path) as src:
        reproject(rasterio.band(src, 1), out, src_nodata=src.nodata, dst_nodata=np.nan,
                  dst_transform=GT, dst_crs=DST_CRS,
                  resampling=Resampling.nearest if categorical else Resampling.bilinear)
    return out

STACK = {f: to_grid(p, f in CATEGORICAL) for f, p in USER_RASTERS.items()}
VALID = MASK & np.all([np.isfinite(STACK[f]) for f in FEATURES], axis=0)
print(f"Grid {W_}×{H_} at {PRED_RES} m; valid pixels {VALID.sum():,} ({VALID.sum()*PIX_KM2:,.0f} km²)")

def predict_grid(stack, model_fn, chunk=250000):
    Xg = pd.DataFrame({f: stack[f][VALID] for f in FEATURES})
    p = np.concatenate([model_fn(Xg.iloc[i:i + chunk]) for i in range(0, len(Xg), chunk)])
    out = np.full((H_, W_), np.nan, np.float32); out[VALID] = p; return out

MODELS = {"RF": lambda d: rf.predict_proba(d[FEATURES])[:, 1], "MaxEnt": me_pred}
PROB = {(m, "Current", "1991-2020"): predict_grid(STACK, fn) for m, fn in MODELS.items()}

CMAP, NORM = ListedColormap(CLASS_COLORS), BoundaryNorm(np.arange(.5, 6.5), 5)
EXTENT = (minx, minx + W_ * PRED_RES, maxy - H_ * PRED_RES, maxy)
LEG = [Patch(color=c, label=f"{k+1} {n}") for k, (n, c) in enumerate(zip(CLASS_NAMES, CLASS_COLORS))]

def base_map_with_coords(ax):
    eastern_dist_utm.boundary.plot(ax=ax, color="grey", lw=0.6)
    eastern_prov_utm.boundary.plot(ax=ax, color="black", lw=1.2)
    for _, r in dist_gdf.iterrows():
        p = r.geometry.representative_point()
        ax.annotate(r.District, (p.x, p.y), ha="center", fontsize=8, alpha=.8)
    ax.tick_params(axis="both", which="major", labelsize=8)
    ax.grid(True, color="gray", linestyle="--", alpha=0.3)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:,.0f} m"))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, p: f"{y:,.0f} m"))

def show_classes(ax, prob, title):
    ax.imshow(np.where(VALID, classify(np.nan_to_num(prob)), np.nan), cmap=CMAP, norm=NORM, extent=EXTENT, interpolation="nearest")
    base_map_with_coords(ax); ax.set_title(title, fontsize=10)

def area_table(prob, model, scen, period):
    cls = classify(np.nan_to_num(prob)); rows = []
    for i, dn in enumerate(DISTRICTS, 1):
        m = VALID & (DIST_ID == i); tot = m.sum() * PIX_KM2
        for k in range(1, 6):
            a = (m & (cls == k)).sum() * PIX_KM2
            rows.append({"Model": model, "Scenario": scen, "Period": period, "District": dn, "Class": k,
                         "Class_Name": CLASS_NAMES[k-1], "Area_km2": round(a, 2), "Percent_of_district": round(100 * a / tot, 2)})
    return pd.DataFrame(rows)

fig, ax = plt.subplots(1, 2, figsize=(18, 10))
for a, m in zip(ax, MODELS):
    show_classes(a, PROB[(m, "Current", "1991-2020")], f"({'ab'[list(MODELS).index(m)]}) {m} – current GWPZ")
fig.legend(handles=LEG, loc="lower center", ncol=5, title=f"GWPZ (probability breaks {PROB_BREAKS})")
plt.tight_layout(rect=(0, .06, 1, 1)); savefig("Fig09_current_GWPZ_RF_MaxEnt"); plt.show()

cur_area = pd.concat([area_table(PROB[(m, "Current", "1991-2020")], m, "Current", "1991-2020") for m in MODELS])
piv = cur_area.pivot_table(index=["Model", "District"], columns="Class_Name", values="Area_km2")[CLASS_NAMES]
print(piv.round(0)); RESULTS["T_Current_Area_by_District"] = cur_area

In [ ]:
# Fig. 10 – area of each class by district; every class area labelled, district total and High+Very High share at the bar end
def fig_area_by_district(cur_area, name="Fig10_area_by_district"):
    TXT = ["white", "black", "black", "black", "white"]
    tot = cur_area[cur_area.Model == "RF"].groupby("District").Area_km2.sum()
    dists = list(tot.sort_values().index)
    fig, axs = plt.subplots(1, 2, figsize=(7.5, 3.8), sharey=True); BH = 0.56; XMAX = 2350
    for ax, m, ttl in [(axs[0], "RF", "(a) Random Forest"), (axs[1], "MaxEnt", "(b) MaxEnt")]:
        s = cur_area[cur_area.Model == m].pivot_table(index="District", columns="Class", values="Area_km2").reindex(dists)
        pc = cur_area[cur_area.Model == m].pivot_table(index="District", columns="Class", values="Percent_of_district").reindex(dists)
        ax.set_xlim(0, XMAX); left = np.zeros(len(dists))
        for k in range(1, 6):
            ax.barh(np.arange(len(dists)), s[k].values, left=left, color=CLASS_COLORS[k-1], edgecolor="white", lw=.6, height=BH, zorder=3)
            left += s[k].values
        fig.canvas.draw()
        mm_per_unit = ax.get_window_extent().width / fig.dpi * 25.4 / XMAX
        for i, dn in enumerate(dists):
            l = 0; small = []
            for k in range(1, 6):
                v = s.loc[dn, k]; lab = f"{v:.0f}"
                if v * mm_per_unit >= len(lab) * 1.25 + .8: ax.text(l + v / 2, i, lab, ha="center", va="center", fontsize=6, color=TXT[k-1], zorder=5)
                else: small.append((l + v / 2, lab, k))
                l += v
            prev = -1e9
            for xc, lab, k in small:                        # small segments: labels above the bar, spread to avoid overlap
                xl = max(xc, prev + (len(lab) * 1.15 + 1.0) / mm_per_unit); prev = xl; yl = i + BH / 2 + .2
                ax.plot([xc, xc, xl], [i + BH / 2, i + BH / 2 + .07, yl - .07], color=CLASS_COLORS[k-1] if k != 3 else "#c9b400", lw=.6, zorder=4)
                ax.text(xl, yl, lab, ha="center", va="center", fontsize=5.4, color="0.15", zorder=5)
            ax.text(l + 25, i, f"{l:,.0f} km²", ha="left", va="center", fontsize=6.3, fontweight="bold", color="0.2")
            ax.text(l + 25, i - .26, f"H+VH {pc.loc[dn, 4] + pc.loc[dn, 5]:.1f}%", ha="left", va="center", fontsize=5.2, color="#1a6e2e")
        ax.set_yticks(range(len(dists)), dists, fontsize=7.5); ax.set_xlabel("Area (km²)")
        ax.set_title(ttl, loc="left", fontweight="bold", fontsize=9); ax.xaxis.grid(True, color="0.88", lw=.5, zorder=0)
        for sp in ("top", "right"): ax.spines[sp].set_visible(False)
        ax.tick_params(axis="y", length=0); ax.set_ylim(-.75, len(dists) - .25)
    fig.legend([Patch(fc=c, ec="0.4", lw=.4) for c in CLASS_COLORS], CLASS_NAMES, loc="lower center", ncol=5, frameon=False, fontsize=7.5,
               title="Groundwater potential zone (class area in km²; H+VH = share of district classed High or Very High)", title_fontsize=7)
    fig.tight_layout(rect=(0, .09, 1, 1), w_pad=2); savefig(name); plt.show()
fig_area_by_district(cur_area)

In [ ]:
# Yield of wells by mapped zone, using out-of-fold (spatial CV) probabilities, independent of training
Wv = data[data.Class == 1].copy(); yt = []
for m, col in [("RF", "OOF_Spatial_RF"), ("MaxEnt", "OOF_Spatial_MaxEnt")]:
    Wv["Zone"] = classify(Wv[col].values)
    rho, p = stats.spearmanr(Wv[col], Wv["yield"])
    groups = [g["yield"].values for _, g in Wv.groupby("Zone") if len(g) > 1]
    kw = stats.kruskal(*groups) if len(groups) > 1 else (np.nan, np.nan)
    for z, g in Wv.groupby("Zone"):
        yt.append({"Model": m, "Mapped zone": CLASS_NAMES[z-1], "n wells": len(g),
                   "Median yield (m3/h)": round(g["yield"].median(), 2), "Mean yield": round(g["yield"].mean(), 2),
                   "Spearman rho (prob vs yield)": round(rho, 3), "Spearman p": round(p, 4), "Kruskal-Wallis p": round(kw[1], 4)})
yt = pd.DataFrame(yt); print(yt.to_string(index=False)); RESULTS["T_Yield_by_Mapped_Zone"] = yt

# Part I – RF–MaxEnt agreement (Fig. 11)

In [ ]:
def agreement(p1, p2):
    c1, c2 = classify(p1[VALID]), classify(p2[VALID])
    s = np.random.default_rng(SEED).choice(len(c1), min(200000, len(c1)), replace=False)
    return {"Exact agreement %": round(100 * np.mean(c1 == c2), 2),
            "Within ±1 class %": round(100 * np.mean(np.abs(c1 - c2) <= 1), 2),
            "Cohen kappa": round(cohen_kappa_score(c1[s], c2[s]), 3),
            "Quadratic weighted kappa": round(cohen_kappa_score(c1[s], c2[s], weights="quadratic"), 3),
            "Pearson r (probabilities)": round(np.corrcoef(p1[VALID], p2[VALID])[0, 1], 3)}

pr, pm = PROB[("RF", "Current", "1991-2020")], PROB[("MaxEnt", "Current", "1991-2020")]
agr_rows = [{"Scenario": "Current", "Period": "1991-2020", **agreement(pr, pm)}]
diff = np.where(VALID, classify(np.nan_to_num(pr)) - classify(np.nan_to_num(pm)), np.nan)
agree_cat = np.where(np.abs(diff) == 0, 1, np.where(np.abs(diff) == 1, 2, 3)).astype(float); agree_cat[~VALID] = np.nan

fig, ax = plt.subplots(1, 2, figsize=(18, 10))
im = ax[0].imshow(diff, cmap="RdBu", vmin=-4, vmax=4, extent=EXTENT, interpolation="nearest")
base_map_with_coords(ax[0]); plt.colorbar(im, ax=ax[0], shrink=.6, label="RF class − MaxEnt class"); ax[0].set_title("(a) Class difference")
ax[1].imshow(agree_cat, cmap=ListedColormap(["#1a9641", "#fdae61", "#d7191c"]), vmin=.5, vmax=3.5, extent=EXTENT, interpolation="nearest")
base_map_with_coords(ax[1])
ax[1].legend(handles=[Patch(color="#1a9641", label="Same class"), Patch(color="#fdae61", label="±1 class"),
                      Patch(color="#d7191c", label="≥2 classes")], loc="lower left")
ax[1].set_title(f"(b) Agreement – exact {agr_rows[0]['Exact agreement %']}%, weighted kappa {agr_rows[0]['Quadratic weighted kappa']}")
plt.tight_layout(); savefig("Fig11_agreement_current"); plt.show()

ag_d = []
for i, dn in enumerate(DISTRICTS, 1):
    m = VALID & (DIST_ID == i)
    ag_d.append({"District": dn, "Same class %": round(100 * np.mean(agree_cat[m] == 1), 1),
                 "±1 class %": round(100 * np.mean(agree_cat[m] == 2), 1), "≥2 classes %": round(100 * np.mean(agree_cat[m] == 3), 1)})
ag_d = pd.DataFrame(ag_d); print(ag_d.to_string(index=False)); RESULTS["T_Agreement_by_District_Current"] = ag_d

# Part J – CMIP6 rainfall scenarios
Only **Rainfall** changes; all other predictors stay at present conditions. Future rainfall is resampled bilinearly to the model grid and bias-corrected by multiplicative delta change: CHIRPS baseline × (CMIP6 ensemble future / WorldClim 1970–2000 baseline).

In [ ]:
rain0 = STACK["Rainfall"]
if not os.path.exists(CLIMATE_BASELINE): raise FileNotFoundError(f"WorldClim baseline needed for the delta change: {CLIMATE_BASELINE}")
base_cm = to_grid(CLIMATE_BASELINE)
missing = [p for p in FUTURE_RAINFALL.values() if not os.path.exists(p)]
if missing: raise FileNotFoundError(f"Upload the CMIP6 rasters: {missing}")
FUT_RAIN, rain_rows = {}, []
for (sc, per), p in FUTURE_RAINFALL.items():
    r = rain0 * to_grid(p) / np.where(base_cm > 0, base_cm, np.nan)
    FUT_RAIN[(sc, per)] = r
    for i, dn in enumerate(DISTRICTS, 1):
        m = VALID & (DIST_ID == i)
        rain_rows.append({"Scenario": sc, "Period": per, "District": dn,
                          "Baseline_mm": round(np.nanmean(rain0[m]), 1), "Future_mm": round(np.nanmean(r[m]), 1),
                          "Change_mm": round(np.nanmean(r[m] - rain0[m]), 1),
                          "Change_%": round(100 * np.nanmean(r[m] - rain0[m]) / np.nanmean(rain0[m]), 1)})
rain_tab = pd.DataFrame(rain_rows); RESULTS["T_Future_Rainfall_by_District"] = rain_tab
print(rain_tab.groupby(["Scenario", "Period"])[["Baseline_mm", "Future_mm", "Change_%"]].mean().round(1))

lo, hi = X_tr["Rainfall"].min(), X_tr["Rainfall"].max(); ext = []
for k, r in FUT_RAIN.items():
    v = r[VALID]; ext.append({"Scenario": k[0], "Period": k[1], "Train rainfall range": f"{lo:.0f}–{hi:.0f}",
                              "% pixels below": round(100 * np.mean(v < lo), 1), "% pixels above": round(100 * np.mean(v > hi), 1)})
ext = pd.DataFrame(ext); print(ext.to_string(index=False)); RESULTS["T_Future_Extrapolation_Check"] = ext

fig, ax = plt.subplots(figsize=(13, 5))
pv = rain_tab.assign(S=rain_tab.Scenario + "\n" + rain_tab.Period).pivot(index="District", columns="S", values="Change_%")
pv.plot.bar(ax=ax, width=.85, colormap="coolwarm_r"); ax.axhline(0, c="k", lw=.8)
ax.set_ylabel("Change in mean annual rainfall (%)"); ax.set_title("Projected rainfall change vs baseline, by district")
ax.legend(ncol=3, fontsize=8); plt.xticks(rotation=0); plt.tight_layout(); savefig("Fig12_rainfall_change"); plt.show()

In [ ]:
for (sc, per), r in FUT_RAIN.items():
    st = dict(STACK); st["Rainfall"] = r
    for m, fn in MODELS.items(): PROB[(m, sc, per)] = predict_grid(st, fn)
SCEN = sorted(FUT_RAIN); PERIODS = sorted({p for _, p in SCEN}); SCS = sorted({s for s, _ in SCEN})

for m in MODELS:                                     # Figs. 13 (MaxEnt) and 14 (RF)
    fig, axs = plt.subplots(len(SCS), len(PERIODS), figsize=(14, 7 * len(SCS)), squeeze=False)
    for i, sc in enumerate(SCS):
        for j, per in enumerate(PERIODS):
            show_classes(axs[i, j], PROB[(m, sc, per)], f"{m} | {sc} | {per}")
    fig.legend(handles=LEG, loc="lower center", ncol=5)
    plt.tight_layout(rect=(0, .04, 1, 1)); savefig(f"Fig{'13' if m == 'MaxEnt' else '14'}_scenarios_{m}"); plt.show()

fut_area = pd.concat([area_table(PROB[(m, sc, per)], m, sc, per) for m in MODELS for sc, per in SCEN])
base = cur_area.rename(columns={"Area_km2": "Baseline_km2", "Percent_of_district": "Baseline_%"})[
       ["Model", "District", "Class", "Class_Name", "Baseline_km2", "Baseline_%"]]
chg = fut_area.merge(base, on=["Model", "District", "Class", "Class_Name"])
chg["Abs_change_km2"] = (chg.Area_km2 - chg.Baseline_km2).round(2)
chg["Change_pp"] = (chg.Percent_of_district - chg["Baseline_%"]).round(2)
chg["Rel_change_%"] = np.where(chg.Baseline_km2 > 0, 100 * chg.Abs_change_km2 / chg.Baseline_km2, np.nan).round(1)
RESULTS["T_Future_Area_Change"] = chg
prov = chg.groupby(["Model", "Scenario", "Period", "Class_Name"])[["Baseline_km2", "Area_km2", "Abs_change_km2"]].sum().reset_index()
prov["Rel_change_%"] = (100 * prov.Abs_change_km2 / prov.Baseline_km2).round(1)
RESULTS["T_Future_Area_Change_Province"] = prov                       # Table 6
print(prov.pivot_table(index=["Scenario", "Period", "Model"], columns="Class_Name", values="Rel_change_%")[CLASS_NAMES])

for sc, per in SCEN:
    agr_rows.append({"Scenario": sc, "Period": per, **agreement(PROB[("RF", sc, per)], PROB[("MaxEnt", sc, per)])})
agr = pd.DataFrame(agr_rows); print(agr.to_string(index=False)); RESULTS["T_Model_Agreement"] = agr

In [ ]:
# District sensitivity (Section 3.4): change in the share of each district classed High or Very High (percentage points),
# over the 12 model × scenario × period combinations; a relative ranking of modeled sensitivity to rainfall, not a forecast
hvh = chg[chg.Class.isin([4, 5])].groupby(["Model", "Scenario", "Period", "District"]).Change_pp.sum().reset_index()
sens = (hvh.groupby("District").Change_pp.agg(mean_change_pp="mean", min_change_pp="min", max_change_pp="max",
                                             n_combinations_with_loss=lambda s: int((s < 0).sum()), n_combinations="count")
        .sort_values("mean_change_pp").round(2))
sens["Relative sensitivity rank (1 = largest modeled loss)"] = range(1, len(sens) + 1)
print(sens.to_string()); RESULTS["T_District_Sensitivity_Runs"] = hvh.round(2); RESULTS["T_District_Sensitivity_Rank"] = sens.reset_index()

# Fig. 15 – mean of RF and MaxEnt change in High + Very High area by district and scenario
hvh["Scenario_Period"] = hvh.Scenario + "\n" + hvh.Period
ens = hvh.groupby(["District", "Scenario_Period"]).Change_pp.mean().unstack()
fig, ax = plt.subplots(figsize=(14, 7))
ens.plot.bar(ax=ax, width=.85, colormap="coolwarm_r", edgecolor="black", lw=.5)
ax.axhline(0, color="black", lw=.8); ax.grid(True, axis="y", ls="--", alpha=.4)
ax.set_title("Mean RF/MaxEnt change in High and Very High GWPZ area by district (modeled response to rainfall only)", fontsize=12)
ax.set_ylabel("Change (percentage points of district area)"); ax.set_xlabel("District"); plt.xticks(rotation=0)
ax.legend(title="Scenario & period", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); savefig("Fig15_district_change_HighVH"); plt.show()

# Appendices A1 (MaxEnt) and B1 (RF): relative change of each class by district and scenario; baseline area in brackets
cols = [f"{s}\n{p}" for s, p in SCEN]
for m, name in [("MaxEnt", "FigA1_MaxEnt_relative_change"), ("RF", "FigB1_RF_relative_change")]:
    fig, axs = plt.subplots(2, 3, figsize=(17, 10)); axs = axs.ravel()
    for k, cn in enumerate(CLASS_NAMES):
        s = chg[(chg.Model == m) & (chg.Class_Name == cn)].assign(S=lambda d: d.Scenario + "\n" + d.Period)
        pv = s.pivot(index="District", columns="S", values="Rel_change_%").reindex(columns=cols)
        bl = s.groupby("District").Baseline_km2.first().reindex(pv.index)
        a = axs[k]; im = a.imshow(pv, cmap="RdBu", vmin=-100, vmax=100, aspect="auto")
        for (r_, c_), v in np.ndenumerate(pv.values): a.text(c_, r_, f"{v:+.0f}", ha="center", va="center", fontsize=7)
        a.set_xticks(range(len(cols)), cols, fontsize=6.5, rotation=45, ha="right")
        a.set_yticks(range(len(pv)), [f"{d} ({b:,.0f} km²)" for d, b in zip(pv.index, bl)], fontsize=7)
        a.set_title(f"{m} – {cn}", fontsize=10)
    axs[5].axis("off"); fig.colorbar(im, ax=axs[5], fraction=.5, label="Relative change in class area (% of baseline)")
    plt.tight_layout(); savefig(name); plt.show()

# Part K – Workflow figure (Fig. 4), software environment and exports

In [ ]:
def workflow_figure():
    MM = 1 / 25.4
    fig = plt.figure(figsize=(190 * MM, 118 * MM)); ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, 190); ax.set_ylim(0, 118); ax.axis("off")
    CF = {"d": "#e8f0fa", "s": "#fff2cc", "m": "#e2efda", "o": "#fbe3d6"}; EC = "#1f3b63"; cols = [6, 52, 98, 144]; Wb, Hb = 40, 19
    for x, t in zip(cols, ["Data", "Sampling & partition", "Modelling & validation", "Mapping & scenarios"]):
        ax.text(x + Wb / 2, 114, t, ha="center", va="center", fontsize=9, fontweight="bold", color=EC); ax.plot([x + 3, x + Wb - 3], [110.5] * 2, color="#8fa6c2", lw=1)
    def box(n, x, y_, title, body, kind, dashed=False):
        ax.add_patch(FancyBboxPatch((x, y_), Wb, Hb, boxstyle="round,pad=0,rounding_size=1.2", fc=CF[kind], ec=EC, lw=.9, ls="--" if dashed else "-"))
        ax.add_patch(Circle((x + 1, y_ + Hb), 2.6, fc="white", ec=EC, lw=.9, zorder=5)); ax.text(x + 1, y_ + Hb, str(n), ha="center", va="center", fontsize=7.5, fontweight="bold", color=EC, zorder=6)
        ax.text(x + Wb / 2, y_ + Hb - 4.2, title, ha="center", va="center", fontsize=7.6, fontweight="bold")
        ax.text(x + Wb / 2, y_ + Hb - 11.2, body, ha="center", va="center", fontsize=6.4, linespacing=1.25)
    rows_ = [86, 61, 36, 9]
    box(1, cols[0], rows_[0], "Borehole inventory (RWB)", "153 records → 152 unique wells\nfive yield classes", "d")
    box(2, cols[0], rows_[1], "12 conditioning factors", "terrain, hydrology, soils, geology,\nLULC, NDVI, rainfall, GWS\nresampled to a common grid", "d")
    box(3, cols[0], rows_[3], "CMIP6 annual precipitation", "SSP1-2.6, SSP2-4.5, SSP5-8.5\n2040–2059 and 2080–2099", "d")
    box(4, cols[1], rows_[0], "Presence–absence sampling", "152 pseudo-absences (RF)\n10,000 background points (MaxEnt)\nidentical predictor extraction", "s")
    box(5, cols[1], rows_[1], "Data partitioning", "stratified 70/30 split (seed 42)\n15 km spatial blocks (5 folds)\ncollinearity screening (ρ, VIF)", "s")
    box(6, cols[1], rows_[3], "Future rainfall layers", "delta-change correction\n(CHIRPS × CMIP6 / WorldClim)\nbilinear resampling to 100 m", "s")
    box(7, cols[2], rows_[0], "Model calibration", "RF: 60-iteration random search\nMaxEnt (elapid): LQHP features,\nβ = 1, cloglog output", "m")
    box(8, cols[2], rows_[1], "Validation (same test set)", "bootstrap 95% CI, paired ΔAUC,\nspatial block CV, calibration,\npermutation importance", "m")
    box(9, cols[2], rows_[2], "Supplementary yield model", "five-class RF; unweighted vs.\ncost-sensitive class weights\n(no resampling)", "m", dashed=True)
    box(10, cols[3], rows_[0], "Current GWPZ maps", "five classes (0.2 intervals)\nRF–MaxEnt agreement map", "o")
    box(11, cols[3], rows_[1], "Scenario GWPZ maps", "only rainfall varied;\nclass-area change, agreement,\nextrapolation check", "o")
    box(12, cols[3], rows_[2], "District sensitivity", "relative ranking by change in\nHigh + Very High share\n→ monitoring priorities", "o")
    K = dict(arrowstyle="-|>", mutation_scale=9, lw=1.0, color="#222222")
    def arr(p, q_, **kw): k = dict(K); k.update(kw); ax.annotate("", xy=q_, xytext=p, arrowprops=k)
    mid = lambda r: r + Hb / 2
    arr((cols[0] + Wb, mid(rows_[0])), (cols[1], mid(rows_[0])))
    ax.plot([cols[0] + Wb, cols[0] + Wb + 4, cols[0] + Wb + 4], [mid(rows_[1]), mid(rows_[1]), mid(rows_[0]) - 4], color="#222", lw=1); arr((cols[0] + Wb + 4, mid(rows_[0]) - 4), (cols[1], mid(rows_[0]) - 4))
    arr((cols[1] + Wb / 2, rows_[0]), (cols[1] + Wb / 2, rows_[1] + Hb)); arr((cols[1] + Wb, mid(rows_[1])), (cols[2], mid(rows_[1])))
    ax.plot([cols[1] + Wb, cols[1] + Wb + 3, cols[1] + Wb + 3], [mid(rows_[1]) + 5, mid(rows_[1]) + 5, mid(rows_[0])], color="#222", lw=1); arr((cols[1] + Wb + 3, mid(rows_[0])), (cols[2], mid(rows_[0])))
    ax.text(cols[1] + Wb + 2.4, mid(rows_[1]) + 11, "70% train", fontsize=6.2, style="italic", ha="right")
    ax.plot([cols[1] + Wb, cols[1] + Wb + 3, cols[1] + Wb + 3], [mid(rows_[1]) - 5, mid(rows_[1]) - 5, mid(rows_[2])], color="#222", lw=1, ls="--"); arr((cols[1] + Wb + 3, mid(rows_[2])), (cols[2], mid(rows_[2])), ls="--")
    ax.text(cols[1] + Wb + 2.4, mid(rows_[2]) + 4, "wells only", fontsize=6.2, style="italic", ha="right")
    arr((cols[2] + Wb / 2, rows_[0]), (cols[2] + Wb / 2, rows_[1] + Hb)); ax.text(cols[2] + Wb / 2 + 1.5, rows_[0] - 3, "fitted models", fontsize=6.2, style="italic")
    arr((cols[2] + Wb, mid(rows_[0])), (cols[3], mid(rows_[0]))); arr((cols[2] + Wb, mid(rows_[1])), (cols[3], mid(rows_[1])))
    arr((cols[3] + Wb / 2, rows_[0]), (cols[3] + Wb / 2, rows_[1] + Hb)); ax.text(cols[3] + Wb / 2 + 1.5, rows_[0] - 3, "baseline", fontsize=6.2, style="italic")
    arr((cols[3] + Wb / 2, rows_[1]), (cols[3] + Wb / 2, rows_[2] + Hb))
    BL = "#2b6cb0"; arr((cols[0] + Wb, mid(rows_[3])), (cols[1], mid(rows_[3])), color=BL)
    ax.plot([cols[1] + Wb, cols[3] - 4, cols[3] - 4], [mid(rows_[3]), mid(rows_[3]), mid(rows_[1]) - 5], color=BL, lw=1); arr((cols[3] - 4, mid(rows_[1]) - 5), (cols[3], mid(rows_[1]) - 5), color=BL)
    ax.text(cols[1] + Wb + 3, mid(rows_[3]) + 2.2, "projected rainfall replaces baseline rainfall", fontsize=6.2, style="italic", color=BL, ha="left")
    for i, (ls, c, t) in enumerate([("-", "#222", "core GWPZ modelling pipeline"), ("--", "#222", "supplementary analysis"), ("-", BL, "climate-scenario pathway")]):
        ax.plot([8, 16], [46 - i * 4.6] * 2, ls=ls, color=c, lw=1.2); ax.text(18, 46 - i * 4.6, t, va="center", fontsize=6.6)
    ax.add_patch(Circle((12, 46 - 3 * 4.6), 2.2, fc="white", ec=EC, lw=.9)); ax.text(12, 46 - 3 * 4.6, "n", ha="center", va="center", fontsize=6.5, fontweight="bold", color=EC); ax.text(18, 46 - 3 * 4.6, "step number", va="center", fontsize=6.6)
    savefig("Fig04_workflow"); plt.show()
workflow_figure()

env = pd.DataFrame({"Item": ["Python", "numpy", "pandas", "geopandas", "rasterio", "scikit-learn", "elapid", "statsmodels",
                             "Random seed", "Prediction grid", "CRS"],
                    "Value": [platform.python_version(), np.__version__, pd.__version__, gpd.__version__, rasterio.__version__,
                              sklearn.__version__, elapid.__version__, __import__("statsmodels").__version__, SEED, f"{PRED_RES} m", DST_CRS]})
RESULTS["T_Software_Environment"] = env; print(env.to_string(index=False))

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

def format_book(path):
    wb = load_workbook(path); thin = Side(style="thin", color="BFBFBF")
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for c in row: c.font = Font(name="Arial", size=10); c.border = Border(top=thin, bottom=thin, left=thin, right=thin)
        for c in ws[1]:
            c.font = Font(name="Arial", size=10, bold=True, color="FFFFFF"); c.fill = PatternFill("solid", fgColor="1F4E78")
            c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        for i, col in enumerate(ws.columns, 1):
            ws.column_dimensions[get_column_letter(i)].width = min(max(10, max(len(str(c.value or "")) for c in col) + 2), 60)
        ws.freeze_panes = "A2"; ws.auto_filter.ref = ws.dimensions
    wb.save(path)

# 1) samples workbook
tcols = ["Sample_ID", "Class", "Class_Label", "Split", "Source", "District", "X", "Y", "Lon", "Lat", "yield", "Yield_Class"] + FEATURES
tab = pd.DataFrame(data[tcols]).rename(columns={"X": "X_UTM35S", "Y": "Y_UTM35S", "yield": "Yield_m3h"})
tab["RF_Prob"] = rf.predict_proba(data[FEATURES])[:, 1].round(4); tab["MaxEnt_Prob"] = me_pred(data[FEATURES]).round(4)
tab["OOF_SpatialCV_RF"] = data.OOF_Spatial_RF.round(4); tab["OOF_SpatialCV_MaxEnt"] = data.OOF_Spatial_MaxEnt.round(4)
for c in CATEGORICAL: tab[c] = tab[c].round().astype("Int64")
p1 = f"{OUT_DIR}/GW_Potential_Training_Samples.xlsx"
with pd.ExcelWriter(p1, engine="openpyxl") as w:
    tab.to_excel(w, sheet_name="All_Samples", index=False)
    tab[tab.Split == "Train"].to_excel(w, sheet_name="Train_70", index=False)
    tab[tab.Split == "Test"].to_excel(w, sheet_name="Test_30", index=False)
    bg[["X", "Y"] + FEATURES].to_excel(w, sheet_name="MaxEnt_Background", index=False)
format_book(p1)

# 2) results workbook (one sheet per table)
p2 = f"{OUT_DIR}/GW_Manuscript_Results.xlsx"
index = pd.DataFrame({"Sheet": list(RESULTS), "Rows": [len(v) for v in RESULTS.values()]})
with pd.ExcelWriter(p2, engine="openpyxl") as w:
    index.to_excel(w, sheet_name="Index", index=False)
    for k, v in RESULTS.items(): pd.DataFrame(v).to_excel(w, sheet_name=k[:31], index=False)
format_book(p2)

# 3) GeoTIFFs of probability and class maps (names as deposited on Zenodo)
prof = dict(driver="GTiff", height=H_, width=W_, count=1, crs=DST_CRS, transform=GT, compress="deflate")
os.makedirs(f"{OUT_DIR}/rasters", exist_ok=True)
for (m, sc, per), p in PROB.items():
    tag = f"{m}_{sc}_{per}".replace(" ", "").replace(".", "")
    with rasterio.open(f"{OUT_DIR}/rasters/{tag}_prob.tif", "w", dtype="float32", nodata=np.nan, **prof) as d: d.write(p, 1)
    cls = np.where(VALID, classify(np.nan_to_num(p)), 0).astype("uint8")
    with rasterio.open(f"{OUT_DIR}/rasters/{tag}_class.tif", "w", dtype="uint8", nodata=0, **prof) as d: d.write(cls, 1)

data[[c for c in tcols if c in data] + ["geometry"]].to_file(f"{OUT_DIR}/GW_training_samples.gpkg", driver="GPKG")
joblib.dump({"rf": rf, "maxent": me, "yield_rf": ym, "features": FEATURES, "prob_breaks": PROB_BREAKS}, f"{OUT_DIR}/models.joblib")
print("✅ outputs:", *sorted(os.listdir(OUT_DIR)), sep="\n  ")

In [ ]:
import shutil
shutil.make_archive("/content/GW_outputs", "zip", OUT_DIR)
if "google.colab" in sys.modules:
    from google.colab import files
    files.download("/content/GW_outputs.zip")